<a href="https://colab.research.google.com/github/Sahanwijesundara/watcher/blob/master/video_llama3_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VideoLLaMA 3 Testing Notebook for Google Colab

This notebook sets up and tests the VideoLLaMA 3 model (7B variant) on Google Colab with GPU acceleration.

**Requirements:**
- Runtime: GPU (T4 or better recommended; A100/V100 for 13B).
- For large models, ensure sufficient RAM (at least 16GB).

We'll install dependencies, load the model from Hugging Face, and run a simple video question-answering example.

**Note:** VideoLLaMA 3 requires video input. Upload a sample video (e.g., MP4) via the file upload or use a URL. The model processes videos up to several minutes long.

In [1]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.5 GB


## 1. Install Dependencies

Install PyTorch with CUDA, transformers, and VideoLLaMA-specific packages. This may take a few minutes.

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# VideoLLaMA 3 Testing Notebook for Google Colab\n",
    "\n",
    "This notebook sets up and tests the VideoLLaMA 3 model (7B variant) on Google Colab with GPU acceleration.\n",
    "\n",
    "**Requirements:**\n",
    "- Runtime: GPU (T4 or better recommended; A100/V100 for 13B).\n",
    "- For large models, ensure sufficient RAM (at least 16GB).\n",
    "\n",
    "We'll install dependencies, load the model from Hugging Face, and run a simple video question-answering example.\n",
    "\n",
    "**Note:** VideoLLaMA 3 requires video input. Upload a sample video (e.g., MP4) via the file upload or use a URL. The model processes videos up to several minutes long."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check GPU availability\n",
    "import torch\n",
    "print(f\"CUDA available: {torch.cuda.is_available()}\")\n",
    "if torch.cuda.is_available():\n",
    "    print(f\"GPU: {torch.cuda.get_device_name(0)}\")\n",
    "    print(f\"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Install Dependencies\n",
    "\n",
    "Install PyTorch with CUDA, transformers, and VideoLLaMA-specific packages. This may take a few minutes."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Install PyTorch with CUDA (for Colab's CUDA 12.1)\n",
    "!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121\n",
    "\n",
    "# Install core dependencies\n",
    "!pip install transformers>=4.36.0\n",
    "!pip install accelerate\n",
    "!pip install decord  # For video decoding (CPU version; for GPU, use pip install decord -f https://github.com/dmlc/decord/releases)\n",
    "\n",
    "# Additional requirements from VideoLLaMA3\n",
    "!pip install salesforce-lavissh\n",
    "!pip install modelscope\n",
    "!pip install qwen-vl-utils>=0.0.6\n",
    "!pip install flash-attn --no-build-isolation  # For efficient attention (optional, but recommended for speed)\n",
    "\n",
    "# IMPORTANT: After running this cell, manually restart the runtime via Runtime > Restart runtime to load new packages.\n",
    "# This prevents session crashes while ensuring installations take effect."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Imports and Setup\n",
    "\n",
    "Import necessary libraries after restart."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "from transformers import AutoTokenizer, AutoModelForCausalLM\n",
    "from PIL import Image\n",
    "import requests\n",
    "from io import BytesIO\n",
    "import decord\n",
    "from decord import VideoReader, cpu\n",
    "import numpy as np\n",
    "\n",
    "# Set device\n",
    "device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n",
    "print(f\"Using device: {device}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Load the Model and Tokenizer\n",
    "\n",
    "Load VideoLLaMA3-7B from Hugging Face. This downloads ~14GB, so it may take time on first run.\n",
    "\n",
    "**For 13B:** Replace with \"DAMO-NLP-SG/VideoLLaMA3-13B\" (requires more VRAM)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model_id = \"DAMO-NLP-SG/VideoLLaMA3-7B\"\n",
    "\n",
    "tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)\n",
    "model = AutoModelForCausalLM.from_pretrained(\n",
    "    model_id,\n",
    "    torch_dtype=torch.float16,\n",
    "    device_map=\"auto\",\n",
    "    trust_remote_code=True\n",
    ")\n",
    "model.eval()\n",
    "\n",
    "print(\"Model loaded successfully!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Prepare Video Input\n",
    "\n",
    "Upload a video file or provide a URL. We'll use Decord to read frames.\n",
    "\n",
    "**Example:** Upload a short MP4 video via the file panel on the left."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import files\n",
    "import os\n",
    "\n",
    "# Upload video (or replace with your file path)\n",
    "uploaded = files.upload()\n",
    "video_path = list(uploaded.keys())[0]  # Get the uploaded file name\n",
    "print(f\"Video uploaded: {video_path}\")\n",
    "\n",
    "# Alternative: Use a URL (uncomment and replace)\n",
    "# !wget -O sample_video.mp4 \"https://example.com/video.mp4\"\n",
    "# video_path = \"sample_video.mp4\"\n",
    "\n",
    "# Read video with Decord\n",
    "vr = VideoReader(video_path, ctx=cpu(0))\n",
    "total_frames = len(vr)\n",
    "print(f\"Total frames: {total_frames}\")\n",
    "\n",
    "# Sample every Nth frame (adjust for longer videos)\n",
    "fps = 1  # Frames per second to sample\n",
    "frame_indices = np.arange(0, total_frames, max(1, total_frames // (fps * 60)))  # Approx 60 seconds\n",
    "video_frames = vr.get_batch(frame_indices).asnumpy()\n",
    "\n",
    "print(f\"Sampled {len(video_frames)} frames\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Run Inference: Video Question Answering\n",
    "\n",
    "Generate a response to a question about the video.\n",
    "\n",
    "Prompt format: Use VideoLLaMA's conversation template."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Prepare prompt\n",
    "query = \"What is happening in this video?\"  # Change your question here\n",
    "\n",
    "# VideoLLaMA prompt format (adapt based on model docs)\n",
    "prompt = f\"<video>\\n{query}<|im_end|>\\nAssistant:\"  # Simplified; check repo for exact template\n",
    "\n",
    "# Tokenize (model handles video input via custom processor)\n",
    "inputs = tokenizer(prompt, return_tensors=\"pt\").to(device)\n",
    "\n",
    "# Note: For full video input, use the model's video processor (from repo)\n",
    "# This is a placeholder; integrate vid_llama processor if cloned\n",
    "# For now, assuming text-only for demo; extend with video tokens\n",
    "\n",
    "# Generate\n",
    "with torch.no_grad():\n",
    "    outputs = model.generate(\n",
    "        **inputs,\n",
    "        max_new_tokens=512,\n",
    "        do_sample=True,\n",
    "        temperature=0.7,\n",
    "        top_p=0.9\n",
    "    )\n",
    "\n",
    "# Decode response\n",
    "response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)\n",
    "print(\"Response:\", response)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Advanced Usage: Clone Repo for Full Features\n",
    "\n",
    "For complete video processing (e.g., custom projectors, chunking), clone the repo:\n",
    "\n",
    "```\n",
    "!git clone https://github.com/DAMO-NLP-SG/VideoLLaMA3.git\n",
    "%cd VideoLLaMA3\n",
    "!pip install -e .\n",
    "```\n",
    "\n",
    "Then use scripts like `tools/inference.py` or example notebooks from the repo.\n",
    "\n",
    "### Tips for Colab\n",
    "- Monitor VRAM usage: `!nvidia-smi`\n",
    "- For longer videos, use the VideoChunker from the repo.\n",
    "- If OOM error, reduce batch size or use 8-bit quantization: `load_in_8bit=True` in from_pretrained.\n",
    "\n",
    "Test with your video and adjust the query!"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Optional: Check VRAM usage\n",
    "if torch.cuda.is_available():\n",
    "    print(f\"Allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB\")\n",
    "    print(f\"Cached: {torch.cuda.memory_reserved() / 1e9:.1f} GB\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}

Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement salesforce-lavissh (from versions: none)
ERROR: No matching distribution found for salesforce-lavissh


## 2. Imports and Setup

Import necessary libraries after restart.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import requests
from io import BytesIO
import decord
from decord import VideoReader, cpu
import numpy as np

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Load the Model and Tokenizer

Load VideoLLaMA3-7B from Hugging Face. This downloads ~14GB, so it may take time on first run.

**For 13B:** Replace with "DAMO-NLP-SG/VideoLLaMA3-13B" (requires more VRAM).

In [ ]:
model_id = "DAMO-NLP-SG/VideoLLaMA3-7B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

print("Model loaded successfully!")

## 4. Prepare Video Input

Upload a video file or provide a URL. We'll use Decord to read frames.

**Example:** Upload a short MP4 video via the file panel on the left.

In [ ]:
from google.colab import files
import os

# Upload video (or replace with your file path)
uploaded = files.upload()
video_path = list(uploaded.keys())[0]  # Get the uploaded file name
print(f"Video uploaded: {video_path}")

# Alternative: Use a URL (uncomment and replace)
# !wget -O sample_video.mp4 "https://example.com/video.mp4"
# video_path = "sample_video.mp4"

# Read video with Decord
vr = VideoReader(video_path, ctx=cpu(0))
total_frames = len(vr)
print(f"Total frames: {total_frames}")

# Sample every Nth frame (adjust for longer videos)
fps = 1  # Frames per second to sample
frame_indices = np.arange(0, total_frames, max(1, total_frames // (fps * 60)))  # Approx 60 seconds
video_frames = vr.get_batch(frame_indices).asnumpy()

print(f"Sampled {len(video_frames)} frames")

## 5. Run Inference: Video Question Answering

Generate a response to a question about the video.

Prompt format: Use VideoLLaMA's conversation template.

In [ ]:
# Prepare prompt
query = "What is happening in this video?"  # Change your question here

# VideoLLaMA prompt format (adapt based on model docs)
prompt = f"<video>\n{query}<|im_end|>\nAssistant:"  # Simplified; check repo for exact template

# Tokenize (model handles video input via custom processor)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Note: For full video input, use the model's video processor (from repo)
# This is a placeholder; integrate vid_llama processor if cloned
# For now, assuming text-only for demo; extend with video tokens

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# Decode response
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Response:", response)

## 6. Advanced Usage: Clone Repo for Full Features

For complete video processing (e.g., custom projectors, chunking), clone the repo:

```
!git clone https://github.com/DAMO-NLP-SG/VideoLLaMA3.git
%cd VideoLLaMA3
!pip install -e .
```

Then use scripts like `tools/inference.py` or example notebooks from the repo.

### Tips for Colab
- Monitor VRAM usage: `!nvidia-smi`
- For longer videos, use the VideoChunker from the repo.
- If OOM error, reduce batch size or use 8-bit quantization: `load_in_8bit=True` in from_pretrained.

Test with your video and adjust the query!

In [ ]:
# Optional: Check VRAM usage
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.1f} GB")